Pré-processamento do Slakh2100

In [5]:
ROOT = "/kaggle/input/datasets/alonhaviv/slakh2100/slakh2100_flac_redux"

config.py

Responsabilidade: única fonte de verdade para parâmetros. Nenhum outro arquivo tem número fixo.

In [6]:
import json
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional


@dataclass
class Config:
    # caminhos
    slakh_root: str = ""
    out_dir: str = ""

    # dados
    split_dirs: dict = field(default_factory=lambda: {"train": "train", "val": "validation", "test": "test"})
    input_class: str = "Bass"
    target_class: str = "Piano"
    n_bars: int = 8
    stride: int = 8                 # saltos entre cada janela
    min_notes: int = 8              # por trilha e janela
    max_win_tokens: int = 768
    beats_per_bar: int = 4          # apenas 4/4
    limit: Optional[int] = None     # nmr maximo de faixas por split (debug)

    # tokenizador
    pitch_range: tuple = (21, 109)  # padrão do miditok
    beat_res: dict = field(default_factory=lambda: {"0-4": 4, "4-8": 2})
    use_velocities: bool = False
    use_programs: bool = False
    use_tempos: bool = False
    use_time_signatures: bool = False
    special_tokens: list = field(default_factory=lambda: ["PAD", "BOS", "EOS", "SEP", "OUT_Piano"])

    # augmentation
    max_transpose: int = 2
    seed: int = 7

    @classmethod
    def from_args(cls, args):
        cfg = cls(slakh_root=args.slakh, out_dir=args.out)
        
        if args.nbars is not None:
            cfg.n_bars = args.nbars
            cfg.stride = args.nbars
            
        if args.stride is not None:
            cfg.stride = args.stride
        
        if args.min_notes is not None:
            cfg.min_notes = args.min_notes
        
        if args.max_len is not None:
            cfg.max_win_tokens = args.max_len
        
        if args.limit is not None:
            cfg.limit = args.limit
        
        return cfg


def tokenizer_kwargs(config):
    """Converte o Config nos argumentos do TokenizerConfig do miditok"""
    beat_res = {}
    for key, value in config.beat_res.items():
        key_items = []
        for x in key.split("-"):
            key_items.append(int(x))
        
        beat_res[tuple(key_items)] = value
    
    return dict(
        pitch_range=tuple(config.pitch_range),
        beat_res=beat_res,
        use_velocities=config.use_velocities,
        use_programs=config.use_programs,
        use_tempos=config.use_tempos,
        use_time_signatures=config.use_time_signatures,
        special_tokens=list(config.special_tokens),
    )


def to_json(config, path):
    with open(path, "w") as f:
        json.dump(asdict(config), f, indent=2)


def from_json(path):
    with open(path) as f:
        data = json.load(f)
    
    data["pitch_range"] = tuple(data["pitch_range"])
    return Config(**data)


tokenization.py

Responsabilidade: tudo que converte entre `Score` e ids. Nada de arquivos, janelas ou hashes.

In [7]:
from miditok import REMI, TokenizerConfig

OUT_TOKEN = "OUT_Piano"


def make_tokenizer(config: Config):
    t_config = TokenizerConfig(**tokenizer_kwargs(config))
    return REMI(t_config)


def save_tokenizer(tok, path):
    tok.save(path)


def load_tokenizer(path):
    return REMI(params=Path(path))


def special_id(tok, name) -> int:
    for chave in (name + "_None", name):
        if chave in tok.vocab:
            return tok.vocab[chave]
    
    especiais = [word for word in tok.vocab if word.endswith("_None") and not word.startswith(("Bar", "Pitch", "Position", "Duration"))]
    raise KeyError(f"token especial '{name}' não encontrado. Candidatos: {especiais} + OUT_*")


def encode_track(tok, score) -> list[int]:
    out = tok(score)
    seq = out[0] if isinstance(out, list) else out
    return list(seq.ids)


def build_sequence(tok, bass_ids, piano_ids):
    """[BOS] baixo [SEP] [OUT_Piano] piano [EOS]  ->  (ids, sep_idx)"""
    
    ids = [special_id(tok, "BOS")] + list(bass_ids)
    ids += [special_id(tok, "SEP"), special_id(tok, OUT_TOKEN)] + list(piano_ids) + [special_id(tok, "EOS")]
    sep_idx = 1 + len(bass_ids)
    return ids, sep_idx


def split_sequence(tok, ids, sep_idx):
    """Inverso de build_sequence: devolve (baixo, piano) sem os especiais."""
    ids = list(ids)
    bos, sep, out, eos = (special_id(tok, n) for n in ("BOS", "SEP", OUT_TOKEN, "EOS"))
    
    if sep_idx is None:
        sep_idx = ids.index(sep)

    assert ids[0] == bos and ids[sep_idx] == sep and ids[sep_idx + 1] == out

    bass = ids[1:sep_idx]
    piano = ids[sep_idx + 2:-1] if ids[-1] == eos else ids[sep_idx + 2:]
    return bass, piano


def decode_track(tok, ids, program, config, is_drum=False):
    specials = {special_id(tok, n) for n in config.special_tokens}
    ids = [i for i in ids if i not in specials]
    return tok([ids], programs=[(program, is_drum)])


def pitch_table(tok):
    """{id: pitch MIDI} e a inversa {pitch MIDI: id}."""
    table = {}
    for name, _id in tok.vocab.items():
        if name.startswith("Pitch_"):
            table[_id] = int(name.split("_")[1])
    
    inverse = {p: i for i, p in table.items()}
    return table, inverse


def transpose_ids(ids, k, table, inverse, pitch_range):
    """Soma k semitons só aos tokens de pitch. Retorna None se algum pitch sair da faixa."""
    low, high = pitch_range
    new = []
    for token in ids:
        if token in table:
            pitch = table[token] + k
            if not (low <= pitch < high) or pitch not in inverse:
                return None
                
            token = inverse[pitch]
        new.append(token)
    
    return new


windows.py

Responsabilidade: ler o Slakh e entregar pares de `Score` já recortados e filtrados, sem tokenizar. Conta o motivo de cada descarte.

In [8]:
import bisect
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import yaml
from symusic import Score


@dataclass
class TrackInfo:
    track_dir: Path
    bass_stem: str
    bass_program: int
    piano_stems: list  # ["S02", "S05", ...]
    piano_programs: list


@dataclass
class Window:
    track: str
    window_idx: int
    piano_stem: str
    bass: Score
    piano: Score
    qpm: float


def stems_of(stems, cls):
    out = []
    for key, value in stems.items():
        if value["inst_class"] == cls:
            out.append(key)

    return sorted(out)


def info_tracks(config, split, limit=None, stats=None):
    root = Path(config.slakh_root) / config.split_dirs[split]
    out = []
    
    for track_dir in sorted(root.glob("Track*")):
        with open(track_dir / "metadata.yaml") as f:
            meta = yaml.safe_load(f)
        
        stems = {key: value for key, value in meta["stems"].items() if value["midi_saved"]}
        bass = stems_of(stems, config.input_class)
        piano = stems_of(stems, config.target_class)

        if not bass or not piano:
            if stats is not None:
                stats["sem_par"] += 1
            continue

        out.append(TrackInfo(
            track_dir=track_dir, 
            bass_stem=bass[0], 
            bass_program=stems[bass[0]]["program_num"], 
            piano_stems=piano, 
            piano_programs=[stems[k]["program_num"] for k in piano]
        ))

        if limit is not None and len(out) >= limit:
            break

    return out


def load_pair(info, piano_stem):
    score_bass = Score(str(info.track_dir / "MIDI" / f"{info.bass_stem}.mid"))
    score_piano = Score(str(info.track_dir / "MIDI" / f"{piano_stem}.mid"))

    if len(score_bass.tracks) == 0 or len(score_piano.tracks) == 0:
        return None, "stem_vazio"

    if score_piano.ticks_per_quarter != score_bass.ticks_per_quarter:
        return None, "tpq_diferente"

    for t in score_bass.time_signatures:
        if t.numerator != 4 or t.denominator != 4:
            return None, "nao_4_4"

    return (score_bass, score_piano), None


def window_starts(score_bass, score_piano, config):
    bar = config.beats_per_bar * score_bass.ticks_per_quarter
    last = max(
        max((n.time for n in score_bass.tracks[0].notes), default=0),  # tempo da última nota de sb
        max((n.time for n in score_piano.tracks[0].notes), default=0)  # tempo da última nota de sp
    )
    n = int(last // bar) - config.n_bars + 1

    return [i * bar for i in range(0, n, config.stride)]


def note_times(score):
    return np.sort(np.array([n.time for n in score.tracks[0].notes], dtype=np.int64))


def count_in(times, s, e):
    # equivale a contar notas com s <= time < e
    return int(np.searchsorted(times, e, side="left") - np.searchsorted(times, s, side="left"))


def tempo_list(score):
    out = []
    for t in score.tempos:
        qpm = getattr(t, "qpm", None)
        if qpm is None:                       # fallback se só existir mspq
            qpm = 60_000_000 / t.mspq
        out.append((t.time, float(qpm)))
    return sorted(out)


def tempo_at(tempos, tick, default=120.0):
    if not tempos:
        return default
    i = bisect.bisect_right([t for t, _ in tempos], tick) - 1
    return tempos[max(i, 0)][1]


def iter_windows(info, config, stats):
    for piano_stem in info.piano_stems:
        par, reason = load_pair(info, piano_stem)

        if par is None:
            stats[reason] += 1
            continue

        score_bass, score_piano = par
        time_bass, time_piano = note_times(score_bass), note_times(score_piano)
        tempos = tempo_list(score_bass)
        bar = config.beats_per_bar * score_bass.ticks_per_quarter

        
        for idx, start in enumerate(window_starts(score_bass, score_piano, config)):
            end = start + config.n_bars * bar
            if (count_in(time_bass, start, end) < config.min_notes
                    or count_in(time_piano, start, end) < config.min_notes):
                stats["poucas_notas"] += 1
                continue
        
            stats["janelas_ok"] += 1
            yield Window(
                track=info.track_dir.name,
                window_idx=idx,
                piano_stem=piano_stem,
                bass=score_bass.clip(start, end).shift_time(-start),
                piano=score_piano.clip(start, end).shift_time(-start),
                qpm=tempo_at(tempos, start),
            )


dedup.py

Responsabilidade: hashes e limpeza. Não conhece tokenizador nem arquivos.

In [9]:
import hashlib
import numpy as np
from collections import Counter, defaultdict


def seq_hash(ids):
    return hashlib.md5(np.asarray(ids, dtype=np.int32).tobytes()).hexdigest()


def piano_hash(ids, sep):
    return seq_hash(ids[sep + 2:-1])  # sem OUT_Piano e sem EOS


def bass_hash(ids, sep):
    return seq_hash(ids[1:sep])  # sem BOS


def add_hashes(examples):
    for ex in examples:
        ex["h_full"] = seq_hash(ex["ids"])
        ex["h_piano"] = piano_hash(ex["ids"], ex["sep_idx"])
        ex["h_bass"] = bass_hash(ex["ids"], ex["sep_idx"])
    return examples


def dedup_within(examples, key="h_full"):
    """Mantém o primeiro exemplo de cada hash."""
    vistos = set()
    kept = []
    for ex in examples:
        if ex[key] not in vistos:
            vistos.add(ex[key])
            kept.append(ex)

    report = {"antes": len(examples), "depois": len(kept), "removidos": len(examples) - len(kept)}
    return kept, report


def classify_groups(examples):
    """Análise dos grupos de exemplos com o mesmo h_full."""
    groups = defaultdict(list)
    for ex in examples:
        groups[ex["h_full"]].append(ex)

    kinds = Counter()
    for g in groups.values():
        if len(g) < 2:
            continue
        if len({ex["track"] for ex in g}) > 1:
            kinds["músicas diferentes"] += 1
        elif len({(ex["track"], ex["window_idx"]) for ex in g}) == 1:
            kinds["mesma música e janela, stems diferentes"] += 1
        else:
            kinds["mesma música, janelas diferentes (repetição)"] += 1
    return dict(kinds)


def contaminated_tracks(examples, ref_hashes, key="h_full"):
    """Faixas que têm ao menos um exemplo cujo hash está em ref_hashes."""
    return {ex["track"] for ex in examples if ex[key] in ref_hashes}


def drop_tracks(examples, tracks):
    """Remove FAIXAS inteiras com janelas contaminadas."""
    return [ex for ex in examples if ex["track"] not in tracks]


def cross_split_check(a, b, key="h_full"):
    """Tamanho da interseção de hashes entre dois splits, deve ser 0."""
    return len({ex[key] for ex in a} & {ex[key] for ex in b})


def make_report(**etapas):
    return {nome: dict(valor) for nome, valor in etapas.items()}
    

dataset.py

Responsabilidade: entregar tensores prontos para o modelo, com máscara de perda e augmentation no treino.

In [10]:
import random
from functools import partial

import torch
from torch.utils.data import Dataset, DataLoader


class SymbolicDataset(Dataset):
    def __init__(self, examples, tok, config, augment: bool):
        self.examples = examples
        self.augment = augment
        self.max_transpose = config.max_transpose
        self.pitch_range = tuple(config.pitch_range)
        self.n_tries = 3
        if augment:
            self.table, self.inverse = pitch_table(tok)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        ex = self.examples[i]
        ids = ex["ids"]
        if self.augment:
            for _ in range(self.n_tries):
                k = random.randint(-self.max_transpose, self.max_transpose)
                if k == 0:
                    break
                
                novo = transpose_ids(ids, k, self.table, self.inverse, self.pitch_range)
                if novo is not None:
                    ids = novo
                    break

        return torch.tensor(ids, dtype=torch.long), ex["sep_idx"]


def worker_init_fn(worker_id):
    random.seed(torch.initial_seed() % 2**32)  # semente por worker, para o k aleatório


def collate(batch, pad_id):
    B = len(batch)
    lengths = torch.tensor([len(ids) for ids, _ in batch])
    L = int(lengths.max())

    x = torch.full((B, L), pad_id, dtype=torch.long)
    loss_mask = torch.zeros((B, L), dtype=torch.bool)
    sep_idx = torch.zeros(B, dtype=torch.long)

    for i, (ids, sep) in enumerate(batch):
        n = len(ids)
        x[i, :n] = ids
        loss_mask[i, sep + 2:n] = True   # piano + EOS
        sep_idx[i] = sep

    pad_mask = torch.arange(L)[None, :] >= lengths[:, None]
    return {"x": x, "loss_mask": loss_mask, "pad_mask": pad_mask, "sep_idx": sep_idx, "lengths": lengths}


def make_loader(examples, tok, config, batch_size, augment, shuffle, num_workers=0):
    ds = SymbolicDataset(examples, tok, config, augment)
    pad_id = special_id(tok, "PAD")
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers,
                      collate_fn=partial(collate, pad_id=pad_id), worker_init_fn=worker_init_fn)


def check_batch(batch, tok, n=3):
    """Sanidade da máscara de perda (use sempre antes de treinar)."""
    x, mask = batch["x"], batch["loss_mask"]
    sep_id, out_id, eos_id = special_id(tok, "SEP"), special_id(tok, OUT_TOKEN), special_id(tok, "EOS")

    assert not (mask & batch["pad_mask"]).any(), "padding entrou na máscara de perda"
    for i in range(min(n, x.shape[0])):
        sep, L = int(batch["sep_idx"][i]), int(batch["lengths"][i])
        assert x[i, sep] == sep_id and x[i, sep + 1] == out_id, "SEP/OUT_Piano fora do lugar"

        alvo = x[i, 1:][mask[i, 1:]]                   # o que o modelo é treinado a prever
        assert alvo[0] == x[i, sep + 2], "o alvo não começa no 1º token do piano"
        assert alvo[-1] == eos_id, "o alvo não termina em EOS"
        assert alvo.numel() == L - (sep + 2), "tamanho do alvo != len(piano)+1"

    print("check_batch OK")

build_data.py

Responsabilidade: orquestrar tudo em um único comando e gravar os artefatos.

In [11]:
import argparse
import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch


def build_parser():
    parser = argparse.ArgumentParser(description="Pré-processador do Slakh2100")
    parser.add_argument("--slakh", required=True, help="Caminho da raiz do dataset (slakh2100_flac_redux)")
    parser.add_argument("--out", required=True, help="Diretório de saída do pré-processamento")
    parser.add_argument("--nbars", type=int, help="Tamanho da janela em compassos")
    parser.add_argument("--stride", type=int, help="Salto entre janelas, em compassos")
    parser.add_argument("--min-notes", type=int, help="Quantidade mínima de notas por trilha em uma janela")
    parser.add_argument("--max-len", type=int, help="Máximo de tokens por exemplo")
    parser.add_argument("--limit", type=int, help="Nº máximo de faixas por split (debug)")
    parser.add_argument("--build-test", action="store_true", help="Constrói também o conjunto de teste")
    return parser

                    
def build_split(config, tok, split):
    stats = Counter()
    examples = []
    
    for info in info_tracks(config, split, limit=config.limit, stats=stats):
        for w in iter_windows(info, config, stats):
            bass = encode_track(tok, w.bass)
            piano = encode_track(tok, w.piano)
            ids, sep = build_sequence(tok, bass, piano)

            if len(ids) > config.max_win_tokens:
                stats["maior_que_max_token"] += 1
                continue

            examples.append({"ids": ids, "sep_idx": sep, "track": w.track, "window_idx": w.window_idx,
                             "piano_stem": w.piano_stem, "qpm": w.qpm})
    
    return examples, stats


def pick_one_per_track(test, seed):
    """Um prompt por faixa, com semente fixa."""
    rng = random.Random(seed)
    per_track = {}
    for ex in test:
        per_track.setdefault(ex["track"], []).append(ex)

    prompts = []
    for track in sorted(per_track):
        ex = rng.choice(per_track[track])
        prompts.append({"track": track, "window_idx": ex["window_idx"], "piano_stem": ex["piano_stem"]})
    
    return prompts


def compute_stats(examples, tok, config):
    if not examples:
        return {"exemplos": 0}

    lens = np.array([len(ex["ids"]) for ex in examples])
    table, _ = pitch_table(tok)
    bar = tok.vocab.get("Bar_None")

    notas_baixo, notas_piano, vazios = [], [], 0
    for ex in examples:
        ids, sep = ex["ids"], ex["sep_idx"]
        baixo, piano = ids[1:sep], ids[sep + 2:-1]
        notas_baixo.append(sum(t in table for t in baixo))
        notas_piano.append(sum(t in table for t in piano))
        vazios += sum(1 for j, t in enumerate(piano)
                      if t == bar and (j + 1 == len(piano) or piano[j + 1] == bar))

    p50, p95, p99 = (float(v) for v in np.percentile(lens, [50, 95, 99]))
    return {
        "faixas": len({ex["track"] for ex in examples}),
        "exemplos": len(examples),
        "tokens_totais": int(lens.sum()),
        "comprimento": {"media": float(lens.mean()), "p50": p50, "p95": p95, "p99": p99, "max": int(lens.max())},
        "notas_por_janela_baixo": {"media": float(np.mean(notas_baixo)), "p5": float(np.percentile(notas_baixo, 5))},
        "notas_por_janela_piano": {"media": float(np.mean(notas_piano)), "p5": float(np.percentile(notas_piano, 5))},
        "fracao_compassos_vazios_piano": vazios / (len(examples) * config.n_bars),
    }


def build_train(config, tok):
    raw, st = build_split(config, tok, "train")
    train, rep = dedup_within(add_hashes(raw))
    return train, {"descartes": dict(st), "dedup": rep, "grupos_duplicados": classify_groups(raw)}


def build_val(config, tok, train):
    raw, st = build_split(config, tok, "val")
    val, rep_dedup = dedup_within(add_hashes(raw))

    h_tr = {ex["h_full"] for ex in train}
    cont = contaminated_tracks(val, h_tr)
    cont_piano = contaminated_tracks(val, {ex["h_piano"] for ex in train}, key="h_piano")
    cont_baixo = contaminated_tracks(val, {ex["h_bass"] for ex in train}, key="h_bass")

    n_antes, n_faixas = len(val), len({ex["track"] for ex in val})
    val = drop_tracks(val, cont)
    assert cross_split_check(train, val) == 0, "vazamento treino -> validação"

    return val, {"descartes": dict(st), "dedup": rep_dedup,
                 "contaminacao": {"faixas": n_faixas, "contaminadas_full": len(cont),
                                  "contaminadas_piano": len(cont_piano), "contaminadas_baixo": len(cont_baixo),
                                  "exemplos_antes": n_antes, "exemplos_depois": len(val)}}


def build_test(config, tok, train, val):
    raw, st = build_split(config, tok, "test")
    test, rep_dedup = dedup_within(add_hashes(raw))

    ref = {ex["h_full"] for ex in train} | {ex["h_full"] for ex in val}
    cont = contaminated_tracks(test, ref)
    n_antes, n_faixas = len(test), len({ex["track"] for ex in test})
    test = drop_tracks(test, cont)
    assert cross_split_check(train, test) == 0 and cross_split_check(val, test) == 0

    prompts = pick_one_per_track(test, config.seed)
    return test, prompts, {"descartes": dict(st), "dedup": rep_dedup,
                           "contaminacao": {"faixas": n_faixas, "contaminadas": len(cont),
                                            "exemplos_antes": n_antes, "exemplos_depois": len(test),
                                            "prompts": len(prompts)}}


def main(argv=None):
    args = build_parser().parse_args(argv)
    config = Config.from_args(args)
    random.seed(config.seed)
    np.random.seed(config.seed)

    out = Path(config.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    to_json(config, out / "config.json")

    tok = make_tokenizer(config)
    save_tokenizer(tok, out / "tokenizer.json")
    print("vocabulário:", len(tok))

    report, stats = {}, {}

    train, report["train"] = build_train(config, tok)
    val, report["val"] = build_val(config, tok, train)
    splits = {"train": train, "val": val}

    if args.build_test:
        test, prompts, report["test"] = build_test(config, tok, train, val)
        splits["test"] = test
        with open(out / "test_prompts.json", "w") as f:
            json.dump(prompts, f, indent=2)

    for nome, exs in splits.items():
        torch.save(exs, out / f"{nome}.pt")
        stats[nome] = compute_stats(exs, tok, config)

    with open(out / "dedup_report.json", "w") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    with open(out / "stats.json", "w") as f:
        json.dump(stats, f, indent=2)

    print(json.dumps(report, indent=2, ensure_ascii=False))
    print(json.dumps(stats, indent=2))

    return splits, tok, config


In [12]:
splits, tok, config = main(["--slakh", ROOT, "--out", "/kaggle/working/data_debug", "--limit", "20"])

vocabulário: 197
{
  "train": {
    "descartes": {
      "sem_par": 3,
      "poucas_notas": 103,
      "janelas_ok": 146,
      "nao_4_4": 5,
      "maior_que_max_token": 3
    },
    "dedup": {
      "antes": 143,
      "depois": 139,
      "removidos": 4
    },
    "grupos_duplicados": {
      "mesma música, janelas diferentes (repetição)": 4
    }
  },
  "val": {
    "descartes": {
      "sem_par": 1,
      "nao_4_4": 9,
      "janelas_ok": 213,
      "poucas_notas": 78,
      "maior_que_max_token": 2
    },
    "dedup": {
      "antes": 211,
      "depois": 192,
      "removidos": 19
    },
    "contaminacao": {
      "faixas": 12,
      "contaminadas_full": 1,
      "contaminadas_piano": 1,
      "contaminadas_baixo": 1,
      "exemplos_antes": 192,
      "exemplos_depois": 181
    }
  }
}
{
  "train": {
    "faixas": 11,
    "exemplos": 139,
    "tokens_totais": 39816,
    "comprimento": {
      "media": 286.44604316546764,
      "p50": 283.0,
      "p95": 535.4,
      "p99": 56

Teste rápido (`--limit 20`), depois o processamento completo.

Testes

In [13]:
# 1) round-trip: ids -> Score -> ids
ex = splits["train"][0]
bass, piano = split_sequence(tok, ex["ids"], ex["sep_idx"])
rb = decode_track(tok, bass, 33, config)
rp = decode_track(tok, piano, 0, config)
print("notas baixo/piano após decode:", len(rb.tracks[0].notes), len(rp.tracks[0].notes))
assert encode_track(tok, rb) == bass and encode_track(tok, rp) == piano, "round-trip dos ids falhou"

# 2) tokenizador recarregado do disco gera os mesmos ids
tok2 = load_tokenizer("/kaggle/working/data_debug/tokenizer.json")
assert encode_track(tok2, rb) == bass
print("vocabulário:", len(tok), len(tok2))

# 3) transposição: só os pitches mudam
table, inverse = pitch_table(tok)
t = transpose_ids(ex["ids"], 2, table, inverse, config.pitch_range)
assert t is not None and len(t) == len(ex["ids"])
assert all((a == b) or (a in table) for a, b in zip(ex["ids"], t))
assert all(table[b] - table[a] == 2 for a, b in zip(ex["ids"], t) if a in table)

# 4) máscara de perda
loader = make_loader(splits["train"], tok, config, batch_size=8, augment=True, shuffle=True)
check_batch(next(iter(loader)), tok)

notas baixo/piano após decode: 18 107
vocabulário: 197 197
check_batch OK


In [14]:
def diagnose_bass_loss(config, tok, split="train", limit=None):
    table, _ = pitch_table(tok)
    lo, hi = config.pitch_range
    bad = []
    for info in info_tracks(config, split, limit=limit):
        for w in iter_windows(info, config, Counter()):
            notes = w.bass.tracks[0].notes
            n_tok = sum(t in table for t in encode_track(tok, w.bass))
            if n_tok < 0.8 * len(notes):
                keys = [(n.time, n.pitch) for n in notes]
                bad.append({
                    "track": w.track, "win": w.window_idx, "midi": len(notes), "tok": n_tok,
                    "fora_da_faixa": sum(not (lo <= n.pitch < hi) for n in notes),
                    "duplicadas": len(keys) - len(set(keys)),
                    "curtas": sum(n.duration <= 1 for n in notes),     # duração em ticks
                })
    print("janelas com perda >20%:", len(bad))
    print("por causa:", {k: sum(b[k] for b in bad) for k in ("fora_da_faixa", "duplicadas", "curtas")})
    print("faixas afetadas:", Counter(b["track"] for b in bad).most_common(10))
    return bad

bad = diagnose_bass_loss(config, tok)

janelas com perda >20%: 0
por causa: {'fora_da_faixa': 0, 'duplicadas': 0, 'curtas': 0}
faixas afetadas: []


In [15]:
def brute_count(info, config):
    total = 0
    for ps in info.piano_stems:
        par, _ = load_pair(info, ps)
        if par is None:
            continue
        sb, sp = par
        nb, np_ = sb.tracks[0].notes, sp.tracks[0].notes
        bar = config.beats_per_bar * sb.ticks_per_quarter
        last = max(max((n.time for n in nb), default=0), max((n.time for n in np_), default=0))
        for i in range(0, int(last // bar) - config.n_bars + 1, config.stride):
            s, e = i * bar, (i + config.n_bars) * bar
            cb = sum(s <= n.time < e for n in nb)
            cp = sum(s <= n.time < e for n in np_)
            if cb >= config.min_notes and cp >= config.min_notes:
                total += 1
    return total

infos = info_tracks(config, "train", limit=30)
a = sum(brute_count(i, config) for i in infos)
b = sum(1 for i in infos for _ in iter_windows(i, config, Counter()))
assert a == b

In [16]:
splits, tok, config = main(["--slakh", ROOT, "--out", "/kaggle/working/debug"])

vocabulário: 197
{
  "train": {
    "descartes": {
      "sem_par": 85,
      "poucas_notas": 4881,
      "janelas_ok": 9576,
      "nao_4_4": 387,
      "maior_que_max_token": 104
    },
    "dedup": {
      "antes": 9472,
      "depois": 7578,
      "removidos": 1894
    },
    "grupos_duplicados": {
      "mesma música, janelas diferentes (repetição)": 384,
      "músicas diferentes": 759,
      "mesma música e janela, stems diferentes": 31
    }
  },
  "val": {
    "descartes": {
      "sem_par": 21,
      "nao_4_4": 92,
      "janelas_ok": 2087,
      "poucas_notas": 991,
      "maior_que_max_token": 20
    },
    "dedup": {
      "antes": 2067,
      "depois": 1893,
      "removidos": 174
    },
    "contaminacao": {
      "faixas": 151,
      "contaminadas_full": 34,
      "contaminadas_piano": 34,
      "contaminadas_baixo": 36,
      "exemplos_antes": 1893,
      "exemplos_depois": 1525
    }
  }
}
{
  "train": {
    "faixas": 626,
    "exemplos": 7578,
    "tokens_totais": 21

In [17]:
splits, tok, config = main(["--slakh", ROOT, "--out", "/kaggle/working/test", "--build-test"])

vocabulário: 197
{
  "train": {
    "descartes": {
      "sem_par": 85,
      "poucas_notas": 4881,
      "janelas_ok": 9576,
      "nao_4_4": 387,
      "maior_que_max_token": 104
    },
    "dedup": {
      "antes": 9472,
      "depois": 7578,
      "removidos": 1894
    },
    "grupos_duplicados": {
      "mesma música, janelas diferentes (repetição)": 384,
      "músicas diferentes": 759,
      "mesma música e janela, stems diferentes": 31
    }
  },
  "val": {
    "descartes": {
      "sem_par": 21,
      "nao_4_4": 92,
      "janelas_ok": 2087,
      "poucas_notas": 991,
      "maior_que_max_token": 20
    },
    "dedup": {
      "antes": 2067,
      "depois": 1893,
      "removidos": 174
    },
    "contaminacao": {
      "faixas": 151,
      "contaminadas_full": 34,
      "contaminadas_piano": 34,
      "contaminadas_baixo": 36,
      "exemplos_antes": 1893,
      "exemplos_depois": 1525
    }
  },
  "test": {
    "descartes": {
      "sem_par": 8,
      "janelas_ok": 1244,
   